# Эксперимент 23: Multilingual Framework

**Статья:** A Multilingual Framework for Dysarthria: Detection, Severity Classification, Speech-to-Text, and Clean Speech Generation (Мультиязычный фреймворк для дизартрии: детекция, классификация степени тяжести, речь-в-текст и генерация чистой речи) 2025

**Ссылка:** [https://arxiv.org/abs/2510.03986](https://arxiv.org/abs/2510.03986)

**Краткое описание модели:** Мультиязычное SSL-представление речи (XLS-R embeddings) с временной агрегацией и классификацией нарушений речи.

**Содержание статьи:** Используется multilingual backbone вместо proxy-CNN: извлекаются эмбеддинги из XLS-R модели и обучается классификатор для детекции дефектов речи в детских скороговорках.

In [ ]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import torch
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, precision_score, recall_score, classification_report
from transformers import AutoModel, AutoFeatureExtractor

exp_dir = Path().resolve()
sys.path.insert(0, str(exp_dir.parent))

from shared import config, data_utils
from shared.results_utils import save_result_csv

## 1. Загрузка данных и разбиение

In [ ]:
paths_train, paths_val, paths_test, y_train, y_val, y_test, letters_train, letters_val, letters_test = data_utils.get_splits()
print(f"Train: {len(paths_train)}, Val: {len(paths_val)}, Test: {len(paths_test)}")
n_letters = letters_train.shape[1]

Train: 1942, Val: 417, Test: 417


## 2. Подготовка признаков / представлений

In [3]:
MODEL_ID = "facebook/wav2vec2-xls-r-300m"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
feature_extractor = AutoFeatureExtractor.from_pretrained(MODEL_ID)
encoder = AutoModel.from_pretrained(MODEL_ID).to(device)
encoder.eval()

def extract_embed(path):
    y, sr = data_utils.load_audio(path, sr=16000, max_sec=config.MAX_DURATION_SEC)
    inp = feature_extractor(y, sampling_rate=16000, return_tensors="pt", padding=True)
    with torch.no_grad():
        hs = encoder(inp.input_values.to(device)).last_hidden_state[0].cpu().numpy()
    return np.concatenate([hs.mean(axis=0), hs.std(axis=0), hs.max(axis=0)], axis=0).astype(np.float32)

X_train = np.stack([extract_embed(p) for p in paths_train])
X_val   = np.stack([extract_embed(p) for p in paths_val])
X_test  = np.stack([extract_embed(p) for p in paths_test])
if letters_train.shape[1] > 0:
    X_train = np.hstack([X_train, letters_train])
    X_val   = np.hstack([X_val, letters_val])
    X_test  = np.hstack([X_test, letters_test])


preprocessor_config.json:   0%|          | 0.00/212 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

I0000 00:00:1774733968.252457   91764 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


pytorch_model.bin:   0%|          | 0.00/1.27G [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.27G [00:00<?, ?B/s]

## 3. Обучение, оценка и сохранение результата

In [ ]:
pipe = Pipeline([("scaler", StandardScaler()), ("clf", SVC(kernel="rbf", class_weight="balanced", probability=True, random_state=config.RANDOM_STATE))])
param_grid = {"clf__C": [0.3, 1.0, 3.0], "clf__gamma": ["scale", "auto"]}
grid = GridSearchCV(pipe, param_grid, cv=5, scoring="f1_macro", refit=True, n_jobs=-1, verbose=1)
grid.fit(X_train, y_train)
clf = grid.best_estimator_

y_pred = clf.predict(X_test)
y_proba = clf.predict_proba(X_test)[:, 1]
accuracy = accuracy_score(y_test, y_pred)
f1_macro = f1_score(y_test, y_pred, average="macro")
f1_bad = f1_score(y_test, y_pred, average="binary", pos_label=config.CLASS_BAD)
roc_auc = roc_auc_score(y_test, y_proba)
precision_bad = precision_score(y_test, y_pred, zero_division=0, pos_label=config.CLASS_BAD)
recall_bad = recall_score(y_test, y_pred, zero_division=0, pos_label=config.CLASS_BAD)

print(classification_report(y_test, y_pred, target_names=config.CLASS_NAMES))
display(pd.DataFrame([{"accuracy": accuracy, "f1_macro": f1_macro, "f1_bad": f1_bad, "roc_auc": roc_auc, "precision_bad": precision_bad, "recall_bad": recall_bad}]))

save_result_csv(
    exp_dir=exp_dir, 
    experiment_id="exp_23_multilingual_framework", 
    experiment_name="Multilingual XLS-R framework", 
    model="XLS-R embedding + SVM", 
    accuracy=accuracy, 
    f1_macro=f1_macro, 
    f1_bad=f1_bad, 
    roc_auc=roc_auc, 
    precision_bad=precision_bad, 
    recall_bad=recall_bad, 
    notes=f"xlsr300m/grid={grid.best_params_}", 
    num_params=None, 
    train_time_sec=None
)

Fitting 5 folds for each of 6 candidates, totalling 30 fits
              precision    recall  f1-score   support

        good       0.82      0.89      0.85       282
         bad       0.72      0.58      0.64       135

    accuracy                           0.79       417
   macro avg       0.77      0.74      0.75       417
weighted avg       0.79      0.79      0.78       417



,accuracy,f1_macro,f1_bad,roc_auc,precision_bad,recall_bad
0,0.791367,0.747384,0.641975,0.83265,0.722222,0.577778


Результат сохранён в result.csv текущего эксперимента
